In [1]:
import pandas as pd 

df = pd.read_csv("./multilabel_classification_emotions.csv")

In [2]:
df

,id,text-clean,text-clean-no-emoji,date,sadness,anger,hopelessness,loneliness,worthlessness,suicide intent,brain dysfunction (forget),emptiness
0,hhcq6e,Found out something awful\n\nMy mum had a boyf...,Found out something awful\nMy mum had a boyfri...,2020-06-28 11:16:59,True,True,True,False,False,False,False,False
1,d0bobn,"I just want to feel wanted ya know?\n\nLike, I...","I just want to feel wanted ya know?\nLike, I h...",2019-09-06 04:10:27,True,True,True,True,False,False,False,True
2,wy400i,Done\n\nI’m writing this as I sit on the side ...,Done\nI'm writing this as I sit on the side of...,2022-08-26 08:53:35,True,True,True,True,True,False,False,True
3,crkjga,"When nobody else celebrates you, learn to cele...","When nobody else celebrates you, learn to cele...",2019-08-17 10:28:21,True,False,False,True,False,False,False,False
4,zq1lwl,goodbye.\n\nI'm done. I have a bottle of jack ...,goodbye.\nI'm done. I have a bottle of jack da...,2022-12-19 19:50:52,False,False,True,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...
6020,lgsz8c,How do you cope with your partners constant wo...,How do you cope with your partners constant wo...,2021-02-10 12:51:18,True,False,True,False,False,False,False,False
6021,10lxasy,Do you feel loved?\n\nDespite your partner’s i...,Do you feel loved?\nDespite your partner's iss...,2023-01-26 17:28:30,True,False,False,False,False,False,False,False
6022,13fxn8l,I’d rather die than work for another 40 years....,I'd rather die than work for another 40 years....,2023-05-12 21:27:24,True,True,True,False,True,True,True,True
6023,179iyj6,What is going on😭😭\n\nSo im 20f and my “friend...,"What is going on\nSo im 20f and my ""friend"" (i...",2023-10-16 22:38:15,True,False,True,True,False,False,False,False


In [3]:
classes = [
    "sadness", 
    "anger", 
    "hopelessness", 
    "loneliness", 
    "worthlessness", 
    "suicide intent", 
    "brain dysfunction (forget)", 
    "emptiness", 
]

import plotly.express as px 

px.imshow(df[classes].corr())

In [4]:
from itertools import combinations
import numpy as np 

def pick_n_most_correlated(df_correlation: pd.DataFrame, n = 2):
    max_corr, best_comb = -1, None
    for comb in combinations(df_correlation.index, n):
        sum_of_correlation = float(np.abs(df_correlation.loc[list(comb), list(comb)].values).sum())
        if sum_of_correlation > max_corr:
            max_corr = sum_of_correlation
            best_comb = comb
    return list(best_comb)

df_correlation = df[classes].corr()
best_comb = pick_n_most_correlated(df_correlation, 3)

sorting_order = []
for i in range(2, len(classes) +1):
    sorting_order += [c for c in pick_n_most_correlated(df_correlation, i) if c not in sorting_order]

print(sorting_order)
px.imshow(df_correlation.loc[sorting_order, sorting_order])


['anger', 'emptiness', 'sadness', 'brain dysfunction (forget)', 'loneliness', 'suicide intent', 'worthlessness', 'hopelessness']


In [5]:
from itertools import combinations
import numpy as np 

def pick_n_less_correlated(df_correlation: pd.DataFrame, n = 2):
    min_corr, best_comb = 100, None
    for comb in combinations(df_correlation.index, n):
        sum_of_correlation = float(np.abs(df_correlation.loc[list(comb), list(comb)].values).sum())
        if sum_of_correlation < min_corr:
            min_corr = sum_of_correlation
            best_comb = comb
    return list(best_comb)

df_correlation = df[classes].corr()
best_comb = pick_n_less_correlated(df_correlation, 3)

sorting_order = []
for i in range(2, len(classes) +1):
    sorting_order += [c for c in pick_n_less_correlated(df_correlation, i) if c not in sorting_order]

print(sorting_order)
px.imshow(df_correlation.loc[sorting_order, sorting_order])


['anger', 'emptiness', 'sadness', 'brain dysfunction (forget)', 'loneliness', 'suicide intent', 'worthlessness', 'hopelessness']


In [12]:
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from torch import Tensor
from torch.nn import Sigmoid
from transformers import EvalPrediction
import numpy as np


/opt/miniconda3/envs/encoder-tuto/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
def multi_label_metrics(results_matrix, labels : Tensor, threshold : float = 0.5
                        ) -> dict:
    '''Taking a results matrix (batch_size x num_labels), the function (with a 
    threshold) associates labels to the results => y_pred
    From this y_pred matrix, evaluate the f1_micro, roc_auc and accuracy metrics
    '''
    # first, apply sigmoid on predictions which are of shape (batch_size, num_labels)
    sigmoid = Sigmoid()
    probs = sigmoid(Tensor(results_matrix))
    # next, use threshold to turn them into integer predictions
    y_pred = np.zeros(probs.shape)
    y_pred[np.where(probs >= threshold)] = 1
    # finally, compute metrics
    y_true = labels
    f1_micro_average = f1_score(y_true=y_true, y_pred=y_pred, average='micro')
    f1_macro_average = f1_score(y_true=y_true, y_pred=y_pred, average='macro')
    roc_auc = roc_auc_score(y_true, y_pred, average = 'micro')
    accuracy = accuracy_score(y_true, y_pred)
    # return as dictionary
    return {'f1_micro': f1_micro_average,
            'f1_macro': f1_macro_average,
             'roc_auc': roc_auc,
             'accuracy': accuracy}

In [14]:
y_true = np.array([
    [1., 0.],
    [0., 0.],
    [1., 1.],
    [1., 1.],
    [0., 1.],
    [1., 1.],
    [1., 0.],
    [0., 0.],
    [1., 1.],
    [1., 1.],
    [0., 0.],
    [0., 1.],
    [1., 1.],
    [1., 0.],
    [0., 1.],
])
y_pred = np.array([
    [1., 0.],
    [0., 0.],
    [0., 1.],
    [1., 1.],
    [1., 1.],
    [1., 1.],
    [1., 1.],
    [1., 0.],
    [0., 0.],
    [0., 0.],
    [0., 1.],
    [1., 1.],
    [1., 0.],
    [1., 1.],
    [0., 1.],
])

In [15]:
multi_label_metrics(y_pred, y_true)

{'f1_micro': 0.75, 'f1_macro': 0.75, 'roc_auc': 0.5, 'accuracy': 0.4}